In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
data=pd.read_csv("Autonomous QUOTE AGENTS.csv")
data.isnull().sum()

Quote_Num                  0
Agent_Type                 0
Q_Creation_DT              0
Q_Valid_DT                 0
Policy_Bind_DT        113757
Region                     0
Agent_Num                  0
Policy_Type                0
HH_Vehicles                0
HH_Drivers                 0
Driver_Age                 0
Driving_Exp                0
Prev_Accidents             0
Prev_Citations             0
Gender                     0
Marital_Status             0
Education                  0
Sal_Range                  0
Coverage                   0
Veh_Usage                  0
Annual_Miles_Range         0
Vehicl_Cost_Range          0
Re_Quote                   0
Quoted_Premium             0
Policy_Bind                0
dtype: int64

In [3]:
data.shape

(146259, 25)

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146259 entries, 0 to 146258
Data columns (total 25 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Quote_Num           146259 non-null  object 
 1   Agent_Type          146259 non-null  object 
 2   Q_Creation_DT       146259 non-null  object 
 3   Q_Valid_DT          146259 non-null  object 
 4   Policy_Bind_DT      32502 non-null   object 
 5   Region              146259 non-null  object 
 6   Agent_Num           146259 non-null  int64  
 7   Policy_Type         146259 non-null  object 
 8   HH_Vehicles         146259 non-null  int64  
 9   HH_Drivers          146259 non-null  int64  
 10  Driver_Age          146259 non-null  int64  
 11  Driving_Exp         146259 non-null  int64  
 12  Prev_Accidents      146259 non-null  int64  
 13  Prev_Citations      146259 non-null  int64  
 14  Gender              146259 non-null  object 
 15  Marital_Status      146259 non-nul

In [5]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="most_frequent")

data["Policy_Bind_DT"] = imputer.fit_transform(data[["Policy_Bind_DT"]]).ravel()

In [6]:
data.isnull().sum()

Quote_Num             0
Agent_Type            0
Q_Creation_DT         0
Q_Valid_DT            0
Policy_Bind_DT        0
Region                0
Agent_Num             0
Policy_Type           0
HH_Vehicles           0
HH_Drivers            0
Driver_Age            0
Driving_Exp           0
Prev_Accidents        0
Prev_Citations        0
Gender                0
Marital_Status        0
Education             0
Sal_Range             0
Coverage              0
Veh_Usage             0
Annual_Miles_Range    0
Vehicl_Cost_Range     0
Re_Quote              0
Quoted_Premium        0
Policy_Bind           0
dtype: int64

In [7]:
data.head(5)

,Quote_Num,Agent_Type,Q_Creation_DT,Q_Valid_DT,Policy_Bind_DT,Region,Agent_Num,Policy_Type,HH_Vehicles,HH_Drivers,...,Marital_Status,Education,Sal_Range,Coverage,Veh_Usage,Annual_Miles_Range,Vehicl_Cost_Range,Re_Quote,Quoted_Premium,Policy_Bind
0,AQ-C-139212,EA,2020/04/25,2020/06/23,2020/05/23,C,2156,Car,3,3,...,Widow,High School,> $ 25 K <= $ 40 K,Balanced,Commute,> 55 K,> $ 10 K <= $ 20 K,No,693.86,Yes
1,AQ-F-136117,EA,2020/02/21,2020/04/20,2020/07/02,F,2153,Van,2,2,...,Dirvorced,Ph.D,> $ 40 K <= $ 60 K,Balanced,Pleasure,> 7.5 K & <= 15 K,<= $ 10 K,No,635.96,No
2,AQ-F-126801,EA,2020/06/19,2020/08/17,2020/07/12,F,2056,Truck,2,1,...,Dirvorced,Ph.D,> $ 40 K <= $ 60 K,Basic,Commute,> 35 K & <= 45 K,> $ 10 K <= $ 20 K,No,780.64,Yes
3,AQ-E-143467,EA,2020/05/02,2020/06/30,2020/05/24,E,2138,Car,1,2,...,Married,Ph.D,> $ 90 K,Basic,Pleasure,<= 7.5 K,<= $ 10 K,No,723.15,Yes
4,AQ-C-143827,EA,2020/02/12,2020/04/11,2020/02/25,C,2327,Truck,3,1,...,Widow,High School,<= $ 25 K,Basic,Pleasure,> 35 K & <= 45 K,<= $ 10 K,No,738.14,Yes


Agent - 1 

In [96]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
data["Veh_Usage"]=le.fit_transform(data["Veh_Usage"])

In [97]:
import re
def convert_miles(x):
    if pd.isna(x):
        return None
    nums=re.findall(r'\d+\.?\d*',x)
    nums=[float(n)*1000 for n in nums]
    if len(nums)==1:
        return nums[0]+5000
    return sum(nums)/len(nums)
data["Annual_Miles_Range"]=data["Annual_Miles_Range"].apply(convert_miles)

In [98]:
from sklearn.cluster import KMeans
kmeans=KMeans(n_clusters=3,random_state=42)
clusters=kmeans.fit_predict(X)
data["Risk_Tier"]=clusters

In [99]:
risk_map={
    0:"Low",
    1:"Medium",
    2:"High"}
data["Risk_Tier"]=data["Risk_Tier"].map(risk_map)

In [100]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
X=data[["Prev_Accidents","Prev_Citations","Driving_Exp","Driver_Age","Veh_Usage","Annual_Miles_Range"]]
sc=StandardScaler()
X=sc.fit_transform(X)
y=data["Risk_Tier"]

In [101]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
model=RandomForestClassifier(n_estimators=200,max_depth=2,oob_score=True)
model.fit(X_train,y_train)

RandomForestClassifier(max_depth=2, n_estimators=200, oob_score=True)

In [102]:
y_pred=model.predict(X_test)
from sklearn.metrics import accuracy_score
print("accuracy score : ",accuracy_score(y_test,y_pred)*100)

accuracy score :  33.53274989744291


Agent-2

In [123]:
from sklearn.preprocessing import LabelEncoder
X=data[["Re_Quote","Q_Valid_DT","Coverage","Agent_Type","Region","Sal_Range","HH_Drivers"]].copy()
le=LabelEncoder()
for col in X.columns:
    X[col]=le.fit_transform(X[col])
y=data["Policy_Bind"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
agent2_model=RandomForestClassifier(n_estimators=300,max_depth=5,oob_score=True)
agent2_model.fit(X_train,y_train)

RandomForestClassifier(max_depth=5, n_estimators=300, oob_score=True)

In [124]:
y_pred=agent2_model.predict(X_test)
print("accuracy : ",accuracy_score(y_test,y_pred))

accuracy :  0.7777929714207575


In [134]:
def predict_conversion(new_data):
    prob=float(agent2_model.predict_proba(new_data)[0][1])
    return{
        "conversion probability":prob
    }

In [135]:
##testing new Data
new_data = pd.DataFrame({
    "Re_Quote":[1],
    "Q_Valid_DT":[1],
    "Coverage":[2],
    "Agent_Type":[1],
    "Region":[3],
    "Sal_Range":[4],
    "HH_Drivers":[2]
})

print(predict_conversion(new_data))

{'conversion probability': 0.22070925647995102}


Agent-3

In [154]:
def agent3_premium_advisor(agent2_model, customer_data, quoted_premium, coverage, salary_range):
    propensity = agent2_model.predict(customer_data)[0]
    if propensity == 0:
        return {
            "Quoted premium is conversion blocker": False,
            "Message": "Low propensity customer"
        }
    if propensity == 1 and row["Policy_Bind"] == 0:
        return{
            "Quoted premium is conversion blocker":True
        }
    if salary_range == "Low":
        limit = 800
    elif salary_range == "Medium":
        limit = 1500
    else:
        limit = 2500
    if quoted_premium > limit:
        if coverage == "Premium":
            new_cov = "Standard"
        else:
            new_cov = "Basic"
        return {
            "Quoted premium is conversion blocker": True,
            "Recommended_Coverage": new_cov,
            "Recommended_Premium_Band": f"Below {limit}"
        }
    return {"Quoted premium is conversion blocker": False}

In [155]:
customer_data = X_test.iloc[[0]]

result = agent3_premium_advisor(
    agent2_model,
    customer_data,
    quoted_premium=2000,
    coverage="Premium",
    salary_range="Medium"
)

print(result)

{'Quoted premium is conversion blocker': True, 'Recommended_Coverage': 'Standard', 'Recommended_Premium_Band': 'Below 1500'}


Agent-4

In [165]:
def agent4_decision_router(agent1_model, agent2_model, sc, customer_row,
                           quoted_premium, coverage, salary_range,
                           agent_type, region):

    risk_features = customer_row[[
        "Prev_Accidents","Prev_Citations","Driving_Exp",
        "Driver_Age","Veh_Usage","Annual_Miles_Range"
    ]]

    risk_scaled = sc.transform(risk_features)
    risk_tier = agent1_model.predict(risk_scaled)[0]

    bind_features = customer_row[[
        "Re_Quote","Q_Valid_DT","Coverage",
        "Agent_Type","Region","Sal_Range","HH_Drivers"
    ]]

    bind_score = agent2_model.predict_proba(bind_features)[0][1]

    premium_result = agent3_premium_advisor(
        agent2_model,
        bind_features,
        quoted_premium,
        coverage,
        salary_range
    )

    premium_flag = premium_result.get("Quoted premium is conversion blocker", False)

    if (risk_tier == "Low") and (bind_score > 0.7) and (not premium_flag):
        decision = "AutoApprove"

    elif premium_flag or (0.4 <= bind_score <= 0.7):
        decision = "AgentFollowUp"

    else:
        decision = "Escalate"

    return {
        "Risk_Tier": risk_tier,
        "Bind_Score": round(bind_score, 3),
        "Premium_Blocker": premium_flag,
        "Decision": decision,
        "Agent_Type": agent_type,
        "Region": region
    }

In [166]:
customer_row = pd.DataFrame({
    "Prev_Accidents":[0],
    "Prev_Citations":[1],
    "Driving_Exp":[8],
    "Driver_Age":[32],
    "Veh_Usage":[1],
    "Annual_Miles_Range":[2],
    "Re_Quote":[0],
    "Q_Valid_DT":[1],
    "Coverage":[2],
    "Agent_Type":[1],
    "Region":[3],
    "Sal_Range":[2],
    "HH_Drivers":[2]
})
quoted_premium = 950
coverage = 2
salary_range = 2
agent_type = "Direct"
region = "West"

In [167]:
result = agent4_decision_router(
    model,
    agent2_model,
    sc,
    customer_row,
    quoted_premium,
    coverage,
    salary_range,
    agent_type,
    region
)

print(result)

{'Risk_Tier': 'Medium', 'Bind_Score': np.float64(0.21), 'Premium_Blocker': False, 'Decision': 'Escalate', 'Agent_Type': 'Direct', 'Region': 'West'}
